In [1]:
import pandas as pd
from nltk.translate.meteor_score import single_meteor_score

In [2]:
dataset = pd.read_excel('Llama-3.2-11B-Vision-Instruct.xlsx')

In [3]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

NameError: name 'rouge_scorer' is not defined

In [ ]:
import jieba
answer_tokens = list(jieba.cut(dataset['answer'].iloc[5]))
model_response_tokens = list(jieba.cut(dataset['model_response_1'].iloc[5]))

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\Jacky\AppData\Local\Temp\jieba.cache
Loading model cost 0.790 seconds.
Prefix dict has been built successfully.


In [9]:
import nltk
reference_tokenized = [nltk.word_tokenize(ref) for ref in dataset['answer'].iloc[5]]
candidate_tokenized = [nltk.word_tokenize(cand) for cand in dataset['model_response_1'].iloc[5]]

In [5]:
from nltk.translate.bleu_score import sentence_bleu

In [6]:
weights = (0.25, 0.25, 0.25)

In [14]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
scores = scorer.score

In [2]:
from pydantic import BaseModel, Config

class _VertexAIBase(BaseModel):
    model_name: str

    class Config:
        protected_namespaces = ()

ImportError: cannot import name 'Config' from 'pydantic' (c:\Users\Jacky\.conda\envs\rag\lib\site-packages\pydantic\__init__.py)

In [21]:
score = scorer.score(dataset['answer'].iloc[5], dataset['model_response_1'].iloc[5])

In [22]:
score

{'rouge1': Score(precision=0.0, recall=0.0, fmeasure=0.0),
 'rougeL': Score(precision=0, recall=0, fmeasure=0)}

In [38]:
dataset['answer'].iloc[0]

'a) 描述組織辨識和評估氣候相關風險的流程。\nb) 描述組織管理氣候相關風險的流程。\nc) 描述如何將識別、評估和管理氣候相關風險的流程整合到組織的整體風險管理中。'

In [39]:
dataset['model_response_1'].iloc[0]

'\n\n根據氣候相關財務揭露(TCFD)的建議，組織應該描述如下：\n\n* 描述組織識別和評估氣候相關風險的過程\n* 描述組織管理氣候相關風險的過程\n* 描述如何將識別、評估和管理氣候相關風險的過程整合到組織的整體風險管理中<|eot_id|>'

In [25]:
dataset.columns

Index(['pdf', 'query', 'answer', 'input_token_1', 'input_token_2',
       'input_token_3', 'output_token_1', 'output_token_2', 'output_token_3',
       'retrieve_context_1', 'retrieve_context_2', 'retrieve_context_3',
       'model_response_1', 'model_response_2', 'model_response_3', 'time_1',
       'time_2', 'time_3', 'Target Context', 'context_precision_1',
       'context_precision_2', 'context_precision_3'],
      dtype='object')

In [26]:
import ast
data = dataset[['query','retrieve_context_1','model_response_1','answer','Target Context']]
data['retrieve_context_1'] = data['retrieve_context_1'].apply(ast.literal_eval)
data.columns = ['question', 'contexts', 'answer', 'ground_truths','reference']

C:\Users\Jacky\AppData\Local\Temp\ipykernel_17768\3593653950.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['retrieve_context_1'] = data['retrieve_context_1'].apply(ast.literal_eval)


In [11]:
from datasets import Dataset

In [27]:
test = Dataset.from_pandas(data)

In [46]:
from rouge_score import rouge_scorer

In [59]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

In [49]:
dataset.columns

Index(['pdf', 'query', 'answer', 'input_token_1', 'input_token_2',
       'input_token_3', 'output_token_1', 'output_token_2', 'output_token_3',
       'retrieve_context_1', 'retrieve_context_2', 'retrieve_context_3',
       'model_response_1', 'model_response_2', 'model_response_3', 'time_1',
       'time_2', 'time_3', 'Target Context', 'context_precision_1',
       'context_precision_2', 'context_precision_3'],
      dtype='object')

In [67]:
from ragas.metrics import Faithfulness

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
from langchain.embeddings import SentenceTransformerEmbeddings
from tqdm import tqdm, trange
embedding_model = SentenceTransformerEmbeddings(model_name='all-MiniLM-L6-v2')

def calculate_similarity(text1, text2):
    embeddings = embedding_model.embed_documents([text1, text2])
    similarity = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    return similarity


C:\Users\Jacky\AppData\Roaming\Python\Python39\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [13]:
scores = []
for id in range(len(response)):
    scores.append(calculate_similarity(response['response'][id],response['answer'][id]))

In [15]:
response['score']=scores
print(sum(scores)/len(scores))

0.6464535556004418


In [20]:
sum(response['times']*response['num_token'])

53100

In [21]:
sum(response['num_token'])

41681